# Gating threshold study: experiments

This notebook finds where to set the welfare gates on the **Personalized** recommender, the study's Standard RS. It runs on the lab server, which holds
the database and the metro networks, and writes small result files to `results/`. Those files are all
that `analysis.ipynb` needs, so the analysis can run anywhere.

**Gates** (after Uğurel & Yabe 2026)
* **PUP** passes a venue only if $\Pr[U \ge 0] = \Phi(\hat U / \sigma_i) \ge \alpha$. Swept over $\alpha = 0.1, 0.2, \dots, 1.0$. $\sigma_i$ is learned from the
  platform's prediction errors $U - \hat U$ on the user's accepted recommendations, shrunk toward a prior $\sigma_0$ that phase 1 calibrates.
* Both gates price travel with $\hat C$, whose value of time $\hat v$ the gate learns per user by Bayesian regression of its prediction errors
  $U - \hat U$ on its own travel-time estimates (`TravelCostEstimator`), never from the like.
* **RM** passes a venue only if its regret $\hat R_k = \max_j \hat U_j - \hat U_k \le \varepsilon$: how far its estimated net utility $\hat U = \hat V - \hat C$
  falls short of the best venue of that type in the whole catalog (the paper's Eq. 6, "the full set A"). The gate considers the recommender's top 10 venues
  and abstains when none is within $\varepsilon$ of the best. Swept over $\varepsilon = 0$ (offer only the catalog's best) and $\varepsilon$ set at the
  10th–90th percentiles of the regret of the venue the ungated recommender offers, as in the paper.

Each (city, seed, threshold) gets one PUP run and one RM run. The controls (No RS, Standard RS = ungated Personalized, and Oracle) run once
per (city, seed) and appear as flat reference lines on the threshold graphs. Size: 8 metros, 2,000 agents, 30 days, 5 seeds.
Oracle is optional (`RUN_ORACLE`): it does not affect the choice of threshold and is by far the slowest arm.

**Order of work.** The RM grid is defined by Standard RS's regrets, so Standard has to finish before the RM sweep starts.
1. Standard RS and No RS, then calibrate the RM grid and PUP's $\sigma_0$ from Standard RS.
2. The PUP sweep and the RM sweep, in one pool.
3. Oracle, the upper-bound reference, when `RUN_ORACLE` is set. It does not affect the choice of threshold and is by far the slowest arm, so it runs last.
4. Export one CSV per city, then write `results/DONE.txt`.

Each finished simulation is cached under `.cache/gating_study/`. An interrupted notebook (for example after a reboot) resumes from where it
stopped. The cache cannot tell that the model has changed, so delete it after any change to the simulation. Run it unattended with `./run_in_background.sh`; progress is appended to `results/progress.log`.

## Setup

In [1]:
import gc
import json
import multiprocessing as mp
import os
import pickle
import time
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd

REPO = Path.cwd().parent

# The server datasource (Postgres: POIs, ACS/PUMS populations, survey personas) is configured in .env.server.
for line in (REPO / ".env.server").read_text().splitlines():
    if line.strip() and not line.lstrip().startswith("#") and "=" in line:
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip().strip('"'))

from welfare_rs import TravelCostEstimator, WelfareAwareOrchestrator, metrics, params
from welfare_rs.datasource import get_datasource
from welfare_rs.experiment_harness import (eval_day_accepted_ids, eval_day_leisure_utilities, table1_metrics,
                                           table2_from_utilities)
from welfare_rs.metro import build_network
from welfare_rs.oracle_rs import oracle_factory
from welfare_rs.recommender_systems import RECOMMENDER_PRESETS, RecommenderConfig
from welfare_rs.simulation import Simulation

In [2]:
# Experiment size
CITIES = ["houston", "la", "chicago", "sf_bay", "nyc", "dc", "seattle", "miami"]
N_AGENTS = 2000
N_DAYS = 30
SEEDS = list(range(5))

# Threshold grids (the paper's ranges)
PUP_ALPHAS = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
# RM: epsilon at percentiles of the regret of the venue Standard RS offers. p0 is epsilon = 0 exactly: offer only the
# catalog's best venue. Finer at the low end, where abstention falls from most evenings to almost none.
RM_PERCENTILES = [0, 10, 20, 30, 40, 50, 60, 75, 90]
# The Oracle upper bound is a reference line only; it does not change which threshold is best.
RUN_ORACLE = True

# The recommender whose gates are tuned: the study's Standard RS.
RECOMMENDER = "Personalized"
BASE_RS = RecommenderConfig(label=RECOMMENDER, **RECOMMENDER_PRESETS[RECOMMENDER])

# Concurrency: up to MAX_PARALLEL runs, one per core. A run starts only if the runs' memory estimates fit in
# the budget and the box still has MIN_AVAILABLE_GB free. Estimates are peak private memory of one run while
# sharing the network, with margin (measured up to 2.2 GB for the other arms and 2.8 GB for Oracle on this model).
MAX_PARALLEL = 62
MEMORY_BUDGET_GB = 180
RUN_MEMORY_GB = {"oracle": 4.0}
DEFAULT_RUN_MEMORY_GB = 3.0
MIN_AVAILABLE_GB = 30

RESULTS_DIR = Path("results")
CACHE_DIR = REPO / ".cache" / "gating_study" / f"{RECOMMENDER.lower()}_{N_AGENTS}agents_{N_DAYS}days"
RESULTS_DIR.mkdir(exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

print(f"{len(CITIES)} cities x {len(SEEDS)} seeds, {N_AGENTS} agents, {N_DAYS} days")

8 cities x 5 seeds, 2000 agents, 30 days


## One shared copy of each metro network

A network is 0.7–3 GB. It is loaded once here, and every run is a forked copy that shares it instead of loading its own, which halves each
run's memory and is what lets one run go on every core.

In [3]:
NETWORKS = {city: build_network(city) for city in CITIES}
gc.collect()
gc.freeze()  # keep the garbage collector from touching (and so copying) the shared networks in every run

## Arms and the single-run function

An `Arm` is one recommender condition. Gated arms wrap the Personalized recommender in the engine's `WelfareAwareOrchestrator`,
wired exactly as the web app wires a gated recommender (`backend/service.py`).

Standard RS goes through the same wrapper in pass-through mode (`"standard"`). It records the regret of every candidate,
which is used to calibrate the RM grid. The pass-through filters nothing and draws no random numbers, so the run is identical to an
unwrapped Personalized recommender.

In [4]:
@dataclass(frozen=True)
class Arm:
    gate: str                           # "no_rs", "oracle", "standard", "pup" or "rm"
    alpha: Optional[float] = None       # PUP: pass a venue only if P(U >= 0) >= alpha
    epsilon: Optional[float] = None     # RM:  pass a venue only if regret <= epsilon
    percentile: Optional[int] = None    # RM:  the regret percentile epsilon was calibrated at

    @property
    def key(self):
        parts = [self.gate]
        if self.percentile is not None:
            parts.append(f"p{self.percentile}")  # two percentiles can calibrate to the same epsilon
        if self.alpha is not None:
            parts.append(f"a{self.alpha:.2f}")
        if self.epsilon is not None:
            parts.append(f"e{self.epsilon:.4f}")
        return "_".join(parts)


class RegretLoggingGate(WelfareAwareOrchestrator):
    # The engine's welfare gate, also recording the regret of the venue it offers (the best-scored survivor). Its
    # travel-cost estimator records the error U - Û of every recommendation taken, which calibrates PUP's sigma prior.
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.regrets = []

    def recommend(self, user, leisure_subtype, top_k_per_system=5):
        passed = super().recommend(user, leisure_subtype, top_k_per_system)
        self.regrets.extend(recs[0].regret for recs in passed.values() if recs)
        return passed


def recommender_factory(arm, sigma_prior=None):
    if arm.gate == "no_rs":
        return None
    if arm.gate == "oracle":
        return oracle_factory("Oracle")
    gate = {"mode": "standard" if arm.gate == "standard" else arm.gate}
    if arm.alpha is not None:
        gate["pup_alpha"] = arm.alpha
    if arm.epsilon is not None:
        gate["rm_epsilon"] = arm.epsilon

    tce = {} if sigma_prior is None else {"sigma_prior": sigma_prior}

    def factory(sim, base_stack):
        # The metro sets the taxi tariff in the gate's travel-cost estimate, as in the web app.
        return RegretLoggingGate(base_stack, TravelCostEstimator(metro=sim.metro, **tce), **gate)
    return factory


def run_one(city, seed, arm, sigma_prior=None):
    # One simulation. Population and network setup mirror the web app's study runner.
    ds = get_datasource()
    population = ds.population(city, replicate=seed % ds.population_replicates(city),
                               limit=N_AGENTS + max(100, N_AGENTS // 5))
    if population is None or len(population) < N_AGENTS:
        raise RuntimeError(f"{city}: no built population with {N_AGENTS} agents")
    sim = Simulation(
        num_agents=N_AGENTS, seed=seed,
        use_recommenders=(arm.gate != "no_rs"),
        persona_rows=ds.personas(),
        persona_csv_path=ds.persona_csv_path() or params.SURVEY_PERSONA_CSV_PATH,
        population_rows=population,
        road_network=NETWORKS[city],
        poi_rows=ds.poi_rows(city),
        disabled_modes=("transit",),
        recommender_config=BASE_RS,
        recommender_factory=recommender_factory(arm, sigma_prior),
    )
    sim.run_days(N_DAYS)
    table1 = table1_metrics(sim)  # paper Table 1, over the evaluation (last) day's leisure trips
    taste = metrics.taste_metrics(sim)  # the same day's leisure visits that matched the agent's latent taste
    return {
        "city": city, "seed": seed, "arm": arm.key, "gate": arm.gate,
        "alpha": arm.alpha, "percentile": arm.percentile, "epsilon": arm.epsilon,
        "mean_utility": table1["mean_utility"], "abstention_rate": table1["abstention_rate"],
        "taste_match": taste["all_rate"], "taste_match_recommended": taste["recommended_rate"],
        "taste_match_organic": taste["organic_rate"],
        "regrets": np.asarray(sim.recommender_stack.regrets, dtype=np.float32) if arm.gate == "standard" else None,
        # Standard's prediction errors: their pooled sum of squares and count calibrate PUP's sigma prior.
        "sq_err": sum(s["sq_err"] for s in sim.recommender_stack.tce.user_estimates.values()) if arm.gate == "standard" else None,
        "n_err": sum(s["n_err"] for s in sim.recommender_stack.tce.user_estimates.values()) if arm.gate == "standard" else None,
        # For Table 2, paired with the No RS run of the same city and seed at export.
        "utils": eval_day_leisure_utilities(sim.agents), "accepted": eval_day_accepted_ids(sim.agents),
    }

## Parallel runner

Each `(city, seed, arm)` simulation runs in its own process, forked from this notebook so that it shares the networks. When a run finishes,
its process exits and the memory goes back to the system. A run is started whenever a core is free and its memory estimate fits.
A failed run is logged and does not stop the others; re-running the notebook retries only the runs that are missing.

In [5]:
def log(message):
    line = f"{datetime.now():%Y-%m-%d %H:%M}  {message}"
    print(line, flush=True)
    with open(RESULTS_DIR / "progress.log", "a") as f:
        f.write(line + "\n")


def available_gb():
    with open("/proc/meminfo") as f:
        for line in f:
            if line.startswith("MemAvailable"):
                return int(line.split()[1]) / 1e6


def run_memory_gb(task):
    return RUN_MEMORY_GB.get(task[2].gate, DEFAULT_RUN_MEMORY_GB)


def can_start(task, running):
    reserved = sum(run_memory_gb(other) for other in running.values())
    return (len(running) < MAX_PARALLEL and reserved + run_memory_gb(task) <= MEMORY_BUDGET_GB
            and available_gb() >= MIN_AVAILABLE_GB)


def cache_path(city, seed, arm):
    return CACHE_DIR / f"{city}_seed{seed}_{arm.key}.pkl"


def run_grid(arms, sigma_prior=None):
    # Run every (city, seed, arm) not yet cached, in the order of `arms`, and return all of them as one row per run.
    tasks = [(city, seed, arm) for arm in arms for city in CITIES for seed in SEEDS]
    pending = [task for task in tasks if not cache_path(*task).exists()]
    n_todo = len(pending)
    log(f"{len(tasks) - n_todo} of {len(tasks)} runs cached; running {n_todo}")

    running, failed, finished = {}, [], 0
    # maxtasksperchild=1: a fresh process per run. fork: runs share the networks and the functions defined here.
    with mp.get_context("fork").Pool(MAX_PARALLEL, maxtasksperchild=1) as pool:
        while pending or running:
            # Start pending runs while cores and memory allow.
            while pending and can_start(pending[0], running):
                task = pending.pop(0)
                running[pool.apply_async(run_one, (*task, sigma_prior))] = task
            time.sleep(10)
            for job in [job for job in running if job.ready()]:
                city, seed, arm = task = running.pop(job)
                finished += 1
                try:
                    result = job.get()
                except Exception as exc:
                    failed.append(task)
                    log(f"[{finished}/{n_todo}] {city} seed {seed} {arm.key}  FAILED: {exc!r}")
                    continue
                path = cache_path(*task)
                tmp = path.with_suffix(".tmp")
                tmp.write_bytes(pickle.dumps(result))
                tmp.replace(path)  # atomic: an interrupted write never leaves a half-written cache file
                log(f"[{finished}/{n_todo}] {city} seed {seed} {arm.key}  U = {result['mean_utility']:+.3f}  "
                    f"abstention = {result['abstention_rate']:.2f}")
    if failed:
        raise RuntimeError(f"{len(failed)} runs failed (see results/progress.log); re-run to retry them")
    return pd.DataFrame([pickle.loads(cache_path(*task).read_bytes()) for task in tasks])

## Phase 1: Standard RS and No RS

In [6]:
NO_RS, STANDARD, ORACLE = Arm("no_rs"), Arm("standard"), Arm("oracle")
phase1 = run_grid([STANDARD, NO_RS])

2026-10-25 01:38  40 of 80 runs cached; running 40


2026-10-25 02:09  [1/40] miami seed 0 standard  U = +0.311  abstention = 0.00


2026-10-25 02:09  [2/40] miami seed 4 standard  U = +0.289  abstention = 0.00


2026-10-25 02:10  [3/40] miami seed 3 standard  U = +0.319  abstention = 0.00


2026-10-25 02:11  [4/40] miami seed 2 standard  U = +0.267  abstention = 0.00


...

2026-10-25 02:33  [36/40] la seed 4 standard  U = +0.265  abstention = 0.00


2026-10-25 02:33  [37/40] la seed 3 standard  U = +0.277  abstention = 0.00


2026-10-25 02:34  [38/40] la seed 0 standard  U = +0.259  abstention = 0.00


2026-10-25 02:34  [39/40] la seed 2 standard  U = +0.231  abstention = 0.00


2026-10-25 02:35  [40/40] la seed 1 standard  U = +0.258  abstention = 0.00


## RM calibration

RM thresholds have no natural scale, so the paper sets $\varepsilon$ at percentiles of the regret that Standard RS's suggestions carry. The percentiles
are taken over every venue it offered, in every city, seed and day, which gives one $\varepsilon$ grid shared by all cities. A lower percentile
is a stricter gate.

PUP's prior uncertainty $\sigma_0$ is calibrated the same way: the root mean squared error of the platform's estimate, $U - \hat U$, over every
recommendation taken under Standard RS.

In [7]:
regrets = np.concatenate(phase1.loc[phase1["gate"] == "standard", "regrets"].to_list())
RM_EPSILONS = {p: round(float(np.percentile(regrets, p)), 4) for p in RM_PERCENTILES}
(RESULTS_DIR / "rm_epsilons.json").write_text(json.dumps(
    {"n_recommendations": len(regrets), "epsilon_by_percentile": RM_EPSILONS}, indent=2))

# PUP's sigma prior: the root mean squared error U - Û over every recommendation taken under Standard RS.
standard = phase1[phase1["gate"] == "standard"]
SIGMA_PRIOR = round(float(np.sqrt(standard["sq_err"].sum() / standard["n_err"].sum())), 4)
(RESULTS_DIR / "pup_sigma.json").write_text(json.dumps(
    {"n_recommendations_taken": int(standard["n_err"].sum()), "sigma_prior": SIGMA_PRIOR}, indent=2))
print(f"PUP sigma prior {SIGMA_PRIOR} from {int(standard['n_err'].sum()):,} accepted Standard-RS recommendations")

print(f"{len(regrets):,} Standard-RS recommendations")
pd.Series(RM_EPSILONS, name="epsilon").rename_axis("percentile").to_frame()

PUP sigma prior 0.3641 from 775,459 accepted Standard-RS recommendations
1,370,567 Standard-RS recommendations


,epsilon
percentile,
0,0.0000
10,0.0000
20,0.0013
30,0.0150
40,0.0311
50,0.0523
60,0.0825
75,0.1687
90,0.4491


## Phase 2: the PUP and RM sweeps

In [8]:
PUP_ARMS = [Arm("pup", alpha=alpha) for alpha in PUP_ALPHAS]
RM_ARMS = [Arm("rm", epsilon=eps, percentile=p) for p, eps in RM_EPSILONS.items()]
phase2 = run_grid([*PUP_ARMS, *RM_ARMS], sigma_prior=SIGMA_PRIOR)

2026-10-25 02:35  0 of 760 runs cached; running 760


2026-10-25 03:10  [1/760] miami seed 2 pup_a0.10  U = +0.278  abstention = 0.00


2026-10-25 03:10  [2/760] miami seed 4 pup_a0.10  U = +0.307  abstention = 0.00


2026-10-25 03:10  [3/760] miami seed 3 pup_a0.10  U = +0.328  abstention = 0.00


2026-10-25 03:10  [4/760] miami seed 1 pup_a0.10  U = +0.307  abstention = 0.00


...

2026-10-25 13:10  [756/760] nyc seed 3 rm_p90_e0.4491  U = +0.226  abstention = 0.00


2026-10-25 13:11  [757/760] nyc seed 0 rm_p90_e0.4491  U = +0.220  abstention = 0.00


2026-10-25 13:11  [758/760] seattle seed 4 rm_p90_e0.4491  U = +0.243  abstention = 0.00


2026-10-25 13:12  [759/760] seattle seed 2 rm_p90_e0.4491  U = +0.238  abstention = 0.00


2026-10-25 13:12  [760/760] nyc seed 4 rm_p90_e0.4491  U = +0.226  abstention = 0.00


## Phase 3: Oracle

The perfect-knowledge upper bound: ranks every venue by the agent's true net utility. A reference line on the graphs only, run when `RUN_ORACLE` is set.

In [9]:
phase3 = run_grid([ORACLE]) if RUN_ORACLE else pd.DataFrame()

2026-10-25 13:13  40 of 40 runs cached; running 0


## Export the results

One CSV per city with one row per run, holding only what the threshold graphs need: the arm and its threshold, plus the run's
evaluation-day mean net utility and abstention rate (paper Table 1 definitions), and the share of that day's leisure visits
that matched the agent's taste (all, recommended and organic), and paper Table 2 against the No RS run of the same city and seed
(harmed %, improved % and mean ORC over the agents who took a recommendation that day). `config.json` records the design. `DONE.txt` is written
last, so it appears only once every result file is complete.

In [10]:
runs = pd.concat([phase1, phase2, phase3], ignore_index=True)

# Paper Table 2 for every run with a recommender, against the No RS run of the same city and seed.
no_rs_runs = runs[runs["gate"] == "no_rs"].set_index(["city", "seed"])
table2 = [
    table2_from_utilities(row.utils, no_rs_runs.loc[(row.city, row.seed), "utils"], accepted=row.accepted)
    if row.gate != "no_rs" else {"harmed_pct": None, "improved_pct": None, "mean_orc": None, "n_matched": None}
    for row in runs.itertuples()
]
runs = pd.concat([runs, pd.DataFrame(table2, index=runs.index)], axis=1)

columns = ["city", "seed", "arm", "gate", "alpha", "percentile", "epsilon", "mean_utility", "abstention_rate",
           "taste_match", "taste_match_recommended", "taste_match_organic",
           "harmed_pct", "improved_pct", "mean_orc", "n_matched"]
for city, city_runs in runs.groupby("city"):
    city_runs = city_runs[columns].astype({"percentile": "Int64", "n_matched": "Int64"})  # integer percentiles, blank where not RM
    city_runs.sort_values(["arm", "seed"]).to_csv(RESULTS_DIR / f"{city}.csv", index=False)

(RESULTS_DIR / "config.json").write_text(json.dumps({
    "cities": CITIES, "n_agents": N_AGENTS, "n_days": N_DAYS, "seeds": SEEDS,
    "recommender": RECOMMENDER, "recommender_config": RECOMMENDER_PRESETS[RECOMMENDER], "pup_alphas": PUP_ALPHAS, "rm_percentiles": RM_PERCENTILES,
    "oracle": RUN_ORACLE, "surprise_weight": params.FEEDBACK_PARAMS["surprise_weight"],
    "pup_sigma_prior": SIGMA_PRIOR, "pup_sigma_prior_strength": params.WELFARE_GATE_DEFAULTS["pup_sigma_prior_strength"],
    # The gate's value of time: Bayesian regression of U - Û on travel time (2026-10-25), not the like.
    "vot_learner": "regression_2026-10-25",
    **{k: params.WELFARE_GATE_DEFAULTS[k] for k in ("vot_prior_mean", "vot_prior_sd", "vot_noise_sd", "vot_bounds")},
}, indent=2))

(RESULTS_DIR / "DONE.txt").write_text(
    f"All experiments finished {datetime.now():%Y-%m-%d %H:%M}.\n"
    f"{len(runs)} runs: {len(CITIES)} cities x {len(SEEDS)} seeds x {runs['arm'].nunique()} arms, "
    f"{N_AGENTS} agents, {N_DAYS} days.\n"
)
log(f"DONE: {len(runs)} runs exported to {RESULTS_DIR.resolve()}")

/tmp/ipykernel_3742409/1096058538.py:1: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  runs = pd.concat([phase1, phase2, phase3], ignore_index=True)


2026-10-25 13:14  DONE: 880 runs exported to /home/airecsim/welfare_recsys_viz/gating_study/results
